# Task MP-DATA-006: Market Identity Canonicalisation, Panel v2, Regime Diagnostics
**Repository:** `ankit-choubey/Mandi_Prediction`  
**Execution Environment:** Kaggle CPU Kernel (Internet Disabled)  
**Evaluated Dataset:** `theankitchoubey/mandi-prediction-audit-003b` (v1 private, unchanged)  
**Objectives:**
1. Inventory all raw market names per district across the 472,385 raw records and evaluate evidence-based alias candidate pairs under the strict Evidence Rule.
2. Execute Pipeline v2 with market canonicalization running before duplicate detection and low-observation filtering, preserving `Market_Raw`.
3. Quantify v1 vs v2 panel disposition and rescued low-observation rows.
4. Re-evaluate Excel (14,741 in-window rows) and CEDA (594 rows) corroboration on v2.
5. Monthly regime diagnostics (2021-09 to 2026-09) with Agmarknet 2.0 milestone markers.
6. Date-alignment testing on the 77 price differences across $d \pm 3$ calendar day windows.
7. Rebuild train-only eligibility and 4-fold rolling origin cross-validation on v2, plus post-migration training feasibility.


In [ ]:
# Cell 1: Environment, Library Versions, Bootstrap Modules, and Input Discovery
import os
import sys
import glob
import json
import base64
import hashlib
import time
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import openpyxl

print("=== ENVIRONMENT & LIBRARY VERSIONS ===")
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"openpyxl: {openpyxl.__version__}")

np.random.seed(42)
print("Random seed initialized to 42.")

# Bootstrap source modules and reference files
working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(".")
src_dir = working_root / "src"
ref_dir = working_root / "data" / "reference"
out005_dir = working_root / "experiments" / "outputs" / "005"

src_dir.mkdir(parents=True, exist_ok=True)
ref_dir.mkdir(parents=True, exist_ok=True)
out005_dir.mkdir(parents=True, exist_ok=True)

if not (src_dir / "__init__.py").exists():
    (src_dir / "__init__.py").write_text("")

if str(working_root.resolve()) not in sys.path:
    sys.path.insert(0, str(working_root.resolve()))
if "/kaggle/working" not in sys.path and os.path.exists("/kaggle"):
    sys.path.insert(0, "/kaggle/working")

b64_payloads = {
    src_dir / "market_identity.py": "IiIiCk1hcmtldCBJZGVudGl0eSBOb3JtYWxpemF0aW9uIGFuZCBFdmlkZW5jZS1CYXNlZCBDYW5vbmljYWxpemF0aW9uIEVuZ2luZS4KClByb3ZpZGVzIGRldGVybWluaXN0aWMsIHB1cmUgZnVuY3Rpb25zIGZvciBBUE1DIG1hbmRpIG1hcmtldCBuYW1lIG5vcm1hbGl6YXRpb24sCmV2aWRlbmNlLWJhc2VkIGFsaWFzIGV2YWx1YXRpb24sIGFsaWFzIHRhYmxlIG1hbmFnZW1lbnQsIGFuZCBEYXRhRnJhbWUKY2Fub25pY2FsaXphdGlvbiB3aXRob3V0IGV4dGVybmFsIEkvTyBzaWRlIGVmZmVjdHMuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3RyaW5nCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUsIFVuaW9uCmltcG9ydCBwYW5kYXMgYXMgcGQKCgojIFJlcXVpcmVkIHNjaGVtYSBmb3IgZGF0YS9yZWZlcmVuY2UvbWFya2V0X2FsaWFzZXMuY3N2Ck1BUktFVF9BTElBU19DT0xVTU5TOiBMaXN0W3N0cl0gPSBbCiAgICAiZGlzdHJpY3QiLAogICAgIm1hcmtldF9yYXciLAogICAgIm1hcmtldF9jYW5vbmljYWwiLAogICAgInJ1bGUiLAogICAgImV2aWRlbmNlIiwKICAgICJmaXJzdF9kYXRlIiwKICAgICJsYXN0X2RhdGUiLAogICAgIm92ZXJsYXBfZGF5cyIsCiAgICAicHJpY2VfYWdyZWVtZW50IiwKICAgICJzdGF0dXMiCl0KCgpkZWYgbm9ybWFsaXplX21hcmtldF9uYW1lKG5hbWU6IE9wdGlvbmFsW3N0cl0pIC0+IHN0cjoKICAgICIiIgogICAgTm9ybWFsaXplcyBhIG1hcmtldCBuYW1lIGludG8gYSBjYW5vbmljYWwgdG9rZW4gc3RyaW5nOgogICAgLSBMb3dlcmNhc2UKICAgIC0gU3RyaXAgcHVuY3R1YXRpb24gKHJlcGxhY2UgcHVuY3R1YXRpb24gbWFya3Mgd2l0aCB3aGl0ZXNwYWNlKQogICAgLSBDb2xsYXBzZSBtdWx0aXBsZSB3aGl0ZXNwYWNlIGNoYXJhY3RlcnMgaW50byBhIHNpbmdsZSBzcGFjZQogICAgLSBSZW1vdmUgdHJhaWxpbmcgJ2FwbWMnIHdvcmQKICAgIC0gU3RyaXAgbGVhZGluZy90cmFpbGluZyB3aGl0ZXNwYWNlCiAgICAKICAgIEV4YW1wbGVzOgogICAgICAgICdHaGF6aWFiYWQgQVBNQycgLT4gJ2doYXppYWJhZCcKICAgICAgICAnS2FucHVyKEdyYWluKSBBUE1DJyAtPiAna2FucHVyIGdyYWluJwogICAgICAgICdKaGFuc2kgKEdyYWluKScgLT4gJ2poYW5zaSBncmFpbicKICAgICAgICAnSGFwdXInIC0+ICdoYXB1cicKICAgICIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UobmFtZSwgc3RyKToKICAgICAgICByZXR1cm4gIiIKICAgIAogICAgIyAxLiBMb3dlcmNhc2UKICAgIHMgPSBuYW1lLnN0cmlwKCkubG93ZXIoKQogICAgaWYgbm90IHM6CiAgICAgICAgcmV0dXJuICIiCiAgICAKICAgICMgMi4gU3RyaXAgcHVuY3R1YXRpb24gKHJlcGxhY2Ugd2l0aCBzcGFjZXMgc28gJ0thbnB1cihHcmFpbiknIC0+ICdrYW5wdXIgZ3JhaW4nKQogICAgcyA9IHJlLnN1YihyIlsiICsgcmUuZXNjYXBlKHN0cmluZy5wdW5jdHVhdGlvbikgKyByIl0iLCAiICIsIHMpCiAgICAKICAgICMgMy4gQ29sbGFwc2UgbXVsdGlwbGUgd2hpdGVzcGFjZQogICAgcyA9IHJlLnN1YihyIlxzKyIsICIgIiwgcykuc3RyaXAoKQogICAgCiAgICAjIDQuIFJlbW92ZSB0cmFpbGluZyAnYXBtYycgd29yZCBib3VuZGFyeQogICAgcyA9IHJlLnN1YihyIlxiYXBtYyQiLCAiIiwgcykuc3RyaXAoKQogICAgCiAgICAjIDUuIEZpbmFsIHdoaXRlc3BhY2UgY29sbGFwc2UKICAgIHMgPSByZS5zdWIociJccysiLCAiICIsIHMpLnN0cmlwKCkKICAgIAogICAgcmV0dXJuIHMKCgpkZWYgZXZhbHVhdGVfYWxpYXNfZXZpZGVuY2UoCiAgICBkaXN0cmljdF8xOiBzdHIsCiAgICBuYW1lXzE6IHN0ciwKICAgIGZpcnN0X2RhdGVfMTogc3RyLAogICAgbGFzdF9kYXRlXzE6IHN0ciwKICAgIGRpc3RyaWN0XzI6IHN0ciwKICAgIG5hbWVfMjogc3RyLAogICAgZmlyc3RfZGF0ZV8yOiBzdHIsCiAgICBsYXN0X2RhdGVfMjogc3RyLAogICAgb3ZlcmxhcHBpbmdfcmVjb3Jkc19kZjogT3B0aW9uYWxbcGQuRGF0YUZyYW1lXSA9IE5vbmUKKSAtPiBUdXBsZVtib29sLCBEaWN0W3N0ciwgQW55XV06CiAgICAiIiIKICAgIEV2YWx1YXRlcyB3aGV0aGVyIHR3byByYXcgbWFya2V0IG5hbWVzIHNhdGlzZnkgdGhlIHN0cmljdCBFdmlkZW5jZSBSdWxlOgogICAgCiAgICBBY2NlcHQgYSBwYWlyIE9OTFkgaWYgQUxMIG9mIHRoZSBmb2xsb3dpbmcgaG9sZDoKICAgICAgMS4gU2FtZSBkaXN0cmljdDogZGlzdHJpY3RfMS5sb3dlcigpID09IGRpc3RyaWN0XzIubG93ZXIoKQogICAgICAyLiBUaGUgbm9ybWFsaXNlZCBuYW1lcyBhcmUgaWRlbnRpY2FsOgogICAgICAgICBub3JtYWxpemVfbWFya2V0X25hbWUobmFtZV8xKSA9PSBub3JtYWxpemVfbWFya2V0X25hbWUobmFtZV8yKQogICAgICAzLiBBTkQgZWl0aGVyOgogICAgICAgICAoYSkgdGhlIHR3byByYXcgbmFtZXMnIGRhdGUgcmFuZ2VzIG92ZXJsYXAgYnkgPD0gNSBjYWxlbmRhciBkYXlzLCBPUgogICAgICAgICAoYikgb24gb3ZlcmxhcHBpbmcgKGNvbW1vZGl0eSwgdmFyaWV0eSwgZ3JhZGUsIGRhdGUpIHJvd3MsIHRoZQogICAgICAgICAgICAgZXhhY3QtcHJpY2UgYWdyZWVtZW50IGlzID49IDk1JS4KICAgICAgICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICAoaXNfYWNjZXB0ZWQ6IGJvb2wsIGV2aWRlbmNlX2RpY3Q6IGRpY3QpCiAgICAiIiIKICAgICMgMS4gRGlzdHJpY3QgY2hlY2sKICAgIGQxID0gc3RyKGRpc3RyaWN0XzEpLnN0cmlwKCkubG93ZXIoKQogICAgZDIgPSBzdHIoZGlzdHJpY3RfMikuc3RyaXAoKS5sb3dlcigpCiAgICBzYW1lX2Rpc3RyaWN0ID0gKGQxID09IGQyKSBhbmQgYm9vbChkMSkKICAgIAogICAgIyAyLiBOb3JtYWxpc2VkIG5hbWUgY2hlY2sKICAgIG5vcm1fMSA9IG5vcm1hbGl6ZV9tYXJrZXRfbmFtZShuYW1lXzEpCiAgICBub3JtXzIgPSBub3JtYWxpemVfbWFya2V0X25hbWUobmFtZV8yKQogICAgc2FtZV9ub3JtID0gKG5vcm1fMSA9PSBub3JtXzIpIGFuZCBib29sKG5vcm1fMSkKICAgIAogICAgIyAzLiBEYXRlIHJhbmdlIG92ZXJsYXAgY2FsY3VsYXRpb24KICAgIGR0MV9zdGFydCA9IHBkLnRvX2RhdGV0aW1lKGZpcnN0X2RhdGVfMSkKICAgIGR0MV9lbmQgPSBwZC50b19kYXRldGltZShsYXN0X2RhdGVfMSkKICAgIGR0Ml9zdGFydCA9IHBkLnRvX2RhdGV0aW1lKGZpcnN0X2RhdGVfMikKICAgIGR0Ml9lbmQgPSBwZC50b19kYXRldGltZShsYXN0X2RhdGVfMikKICAgIAogICAgb3ZlcmxhcF9zdGFydCA9IG1heChkdDFfc3RhcnQsIGR0Ml9zdGFydCkKICAgIG92ZXJsYXBfZW5kID0gbWluKGR0MV9lbmQsIGR0Ml9lbmQpCiAgICAKICAgIGlmIG92ZXJsYXBfZW5kID49IG92ZXJsYXBfc3RhcnQ6CiAgICAgICAgb3ZlcmxhcF9kYXlzID0gaW50KChvdmVybGFwX2VuZCAtIG92ZXJsYXBfc3RhcnQpLmRheXMpICsgMQogICAgZWxzZToKICAgICAgICBvdmVybGFwX2RheXMgPSAwCiAgICAgICAgCiAgICBwcmljZV9hZ3JlZW1lbnQ6IE9wdGlvbmFsW2Zsb2F0XSA9IE5vbmUKICAgIHJ1bGVfcGFzc2VkOiBPcHRpb25hbFtzdHJdID0gTm9uZQogICAgZXZpZGVuY2Vfc3RyID0gIiIKICAgIHN0YXR1cyA9ICJVTk1BUFBFRC9SRVZJRVciCiAgICAKICAgIGlmIG5vdCBzYW1lX2Rpc3RyaWN0OgogICAgICAgIHJldHVybiBGYWxzZSwgewogICAgICAgICAgICAic3RhdHVzIjogIlVOTUFQUEVEL1JFVklFVyIsCiAgICAgICAgICAgICJydWxlIjogIkZhaWxlZCAoZGlmZmVyZW50IGRpc3RyaWN0cykiLAogICAgICAgICAgICAiZXZpZGVuY2UiOiBmIkRpc3RyaWN0IG1pc21hdGNoOiAne2Rpc3RyaWN0XzF9JyB2cyAne2Rpc3RyaWN0XzJ9JyIsCiAgICAgICAgICAgICJvdmVybGFwX2RheXMiOiBvdmVybGFwX2RheXMsCiAgICAgICAgICAgICJwcmljZV9hZ3JlZW1lbnQiOiBOb25lCiAgICAgICAgfQogICAgICAgIAogICAgaWYgbm90IHNhbWVfbm9ybToKICAgICAgICByZXR1cm4gRmFsc2UsIHsKICAgICAgICAgICAgInN0YXR1cyI6ICJVTk1BUFBFRC9SRVZJRVciLAogICAgICAgICAgICAicnVsZSI6ICJGYWlsZWQgKG5vcm1hbGlzZWQgbmFtZXMgbWlzbWF0Y2gpIiwKICAgICAgICAgICAgImV2aWRlbmNlIjogZiJOb3JtYWxpemVkIG1pc21hdGNoOiAne25vcm1fMX0nIHZzICd7bm9ybV8yfSciLAogICAgICAgICAgICAib3ZlcmxhcF9kYXlzIjogb3ZlcmxhcF9kYXlzLAogICAgICAgICAgICAicHJpY2VfYWdyZWVtZW50IjogTm9uZQogICAgICAgIH0KICAgICAgICAKICAgICMgQ29uZGl0aW9uIChhKTogT3ZlcmxhcCA8PSA1IGRheXMgKGUuZy4gY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uKQogICAgaWYgb3ZlcmxhcF9kYXlzIDw9IDU6CiAgICAgICAgcnVsZV9wYXNzZWQgPSAiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIgogICAgICAgIHN0YXR1cyA9ICJBQ0NFUFRFRCIKICAgICAgICBldmlkZW5jZV9zdHIgPSBmIm92ZXJsYXBfZGF5cz17b3ZlcmxhcF9kYXlzfSA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbikiCiAgICBlbHNlOgogICAgICAgICMgQ29uZGl0aW9uIChiKTogUHJpY2UgYWdyZWVtZW50IG9uIG92ZXJsYXBwaW5nIHJlY29yZHMKICAgICAgICBpZiBvdmVybGFwcGluZ19yZWNvcmRzX2RmIGlzIG5vdCBOb25lIGFuZCBub3Qgb3ZlcmxhcHBpbmdfcmVjb3Jkc19kZi5lbXB0eToKICAgICAgICAgICAgdG90YWxfbWF0Y2hlcyA9IGxlbihvdmVybGFwcGluZ19yZWNvcmRzX2RmKQogICAgICAgICAgICBleGFjdF9tYXRjaGVzID0gaW50KChvdmVybGFwcGluZ19yZWNvcmRzX2RmWyJwcmljZV8xIl0gPT0gb3ZlcmxhcHBpbmdfcmVjb3Jkc19kZlsicHJpY2VfMiJdKS5zdW0oKSkKICAgICAgICAgICAgcHJpY2VfYWdyZWVtZW50ID0gcm91bmQoZmxvYXQoZXhhY3RfbWF0Y2hlcyAvIHRvdGFsX21hdGNoZXMpLCA0KSBpZiB0b3RhbF9tYXRjaGVzID4gMCBlbHNlIDAuMAogICAgICAgICAgICAKICAgICAgICAgICAgaWYgcHJpY2VfYWdyZWVtZW50ID49IDAuOTU6CiAgICAgICAgICAgICAgICBydWxlX3Bhc3NlZCA9ICJSdWxlIEIgKG92ZXJsYXBwaW5nIGNvbmNvcmRhbmNlID49IDk1JSkiCiAgICAgICAgICAgICAgICBzdGF0dXMgPSAiQUNDRVBURUQiCiAgICAgICAgICAgICAgICBldmlkZW5jZV9zdHIgPSBmIm92ZXJsYXBfZGF5cz17b3ZlcmxhcF9kYXlzfSA+IDU7IHByaWNlX2FncmVlbWVudD17cHJpY2VfYWdyZWVtZW50Oi40Zn0gPj0gMC45NSAoe2V4YWN0X21hdGNoZXN9L3t0b3RhbF9tYXRjaGVzfSkiCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBydWxlX3Bhc3NlZCA9ICJGYWlsZWQgUnVsZSBCIChwcmljZSBhZ3JlZW1lbnQgPCA5NSUpIgogICAgICAgICAgICAgICAgc3RhdHVzID0gIlVOTUFQUEVEL1JFVklFVyIKICAgICAgICAgICAgICAgIGV2aWRlbmNlX3N0ciA9IGYib3ZlcmxhcF9kYXlzPXtvdmVybGFwX2RheXN9ID4gNTsgcHJpY2VfYWdyZWVtZW50PXtwcmljZV9hZ3JlZW1lbnQ6LjRmfSA8IDAuOTUgKHtleGFjdF9tYXRjaGVzfS97dG90YWxfbWF0Y2hlc30pIgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJ1bGVfcGFzc2VkID0gIkZhaWxlZCBSdWxlIEIgKG5vIG92ZXJsYXAgcHJpY2UgcmVjb3JkcykiCiAgICAgICAgICAgIHN0YXR1cyA9ICJVTk1BUFBFRC9SRVZJRVciCiAgICAgICAgICAgIGV2aWRlbmNlX3N0ciA9IGYib3ZlcmxhcF9kYXlzPXtvdmVybGFwX2RheXN9ID4gNTsgbWlzc2luZyBvdmVybGFwcGluZyBwcmljZSBkYXRhIGZvciB2ZXJpZmljYXRpb24iCiAgICAgICAgICAgIAogICAgaXNfYWNjZXB0ZWQgPSAoc3RhdHVzID09ICJBQ0NFUFRFRCIpCiAgICByZXR1cm4gaXNfYWNjZXB0ZWQsIHsKICAgICAgICAic3RhdHVzIjogc3RhdHVzLAogICAgICAgICJydWxlIjogcnVsZV9wYXNzZWQsCiAgICAgICAgImV2aWRlbmNlIjogZXZpZGVuY2Vfc3RyLAogICAgICAgICJvdmVybGFwX2RheXMiOiBvdmVybGFwX2RheXMsCiAgICAgICAgInByaWNlX2FncmVlbWVudCI6IHByaWNlX2FncmVlbWVudAogICAgfQoKCmRlZiBsb2FkX21hcmtldF9hbGlhc2VzKGNzdl9wYXRoOiBVbmlvbltzdHIsIFBhdGhdKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiIKICAgIExvYWRzIGFuZCB2YWxpZGF0ZXMgdGhlIG1hcmtldCBhbGlhcyByZWZlcmVuY2UgdGFibGUgZnJvbSBDU1YuCiAgICBFbnN1cmVzIGFsbCByZXF1aXJlZCBjb2x1bW5zIGFyZSBwcmVzZW50IGFuZCBzdHJpcHMgd2hpdGVzcGFjZS4KICAgICIiIgogICAgcGF0aF9vYmogPSBQYXRoKGNzdl9wYXRoKQogICAgaWYgbm90IHBhdGhfb2JqLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWFya2V0IGFsaWFzZXMgcmVmZXJlbmNlIGZpbGUgbm90IGZvdW5kOiB7cGF0aF9vYmp9IikKICAgICAgICAKICAgIGRmID0gcGQucmVhZF9jc3YocGF0aF9vYmopCiAgICBtaXNzaW5nX2NvbHMgPSBbYyBmb3IgYyBpbiBNQVJLRVRfQUxJQVNfQ09MVU1OUyBpZiBjIG5vdCBpbiBkZi5jb2x1bW5zXQogICAgaWYgbWlzc2luZ19jb2xzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNYXJrZXQgYWxpYXNlcyBDU1YgaXMgbWlzc2luZyByZXF1aXJlZCBjb2x1bW5zOiB7bWlzc2luZ19jb2xzfSIpCiAgICAgICAgCiAgICAjIENsZWFuIHdoaXRlc3BhY2Ugb24gc3RyaW5nIGNvbHVtbnMKICAgIGZvciBjIGluIFsiZGlzdHJpY3QiLCAibWFya2V0X3JhdyIsICJtYXJrZXRfY2Fub25pY2FsIiwgInJ1bGUiLCAic3RhdHVzIl06CiAgICAgICAgaWYgYyBpbiBkZi5jb2x1bW5zOgogICAgICAgICAgICBkZltjXSA9IGRmW2NdLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpCiAgICAgICAgICAgIAogICAgcmV0dXJuIGRmCgoKZGVmIGJ1aWxkX2FsaWFzX21hcHBpbmcoYWxpYXNfZGY6IHBkLkRhdGFGcmFtZSkgLT4gRGljdFtUdXBsZVtzdHIsIHN0cl0sIHN0cl06CiAgICAiIiIKICAgIEJ1aWxkcyBhIGxvb2t1cCBtYXBwaW5nIGZyb20gYWNjZXB0ZWQgbWFya2V0IGFsaWFzZXM6CiAgICAgICAgKGRpc3RyaWN0X2xvd2VyLCByYXdfbWFya2V0X2xvd2VyKSAtPiBjYW5vbmljYWxfbWFya2V0X25hbWUKICAgIE9ubHkgcmVjb3JkcyB3aXRoIHN0YXR1cyA9PSAnQUNDRVBURUQnIGFyZSBpbmNsdWRlZC4KICAgICIiIgogICAgbWFwcGluZzogRGljdFtUdXBsZVtzdHIsIHN0cl0sIHN0cl0gPSB7fQogICAgYWNjZXB0ZWRfZGYgPSBhbGlhc19kZlthbGlhc19kZlsic3RhdHVzIl0uc3RyLnVwcGVyKCkgPT0gIkFDQ0VQVEVEIl0KICAgIAogICAgZm9yIF8sIHJvdyBpbiBhY2NlcHRlZF9kZi5pdGVycm93cygpOgogICAgICAgIGtleSA9IChyb3dbImRpc3RyaWN0Il0uc3RyaXAoKS5sb3dlcigpLCByb3dbIm1hcmtldF9yYXciXS5zdHJpcCgpLmxvd2VyKCkpCiAgICAgICAgbWFwcGluZ1trZXldID0gcm93WyJtYXJrZXRfY2Fub25pY2FsIl0uc3RyaXAoKQogICAgICAgIAogICAgcmV0dXJuIG1hcHBpbmcKCgpkZWYgYXBwbHlfbWFya2V0X2Nhbm9uaWNhbGl6YXRpb24oCiAgICBkZjogcGQuRGF0YUZyYW1lLAogICAgYWxpYXNfc291cmNlOiBVbmlvbltwZC5EYXRhRnJhbWUsIERpY3RbVHVwbGVbc3RyLCBzdHJdLCBzdHJdLCBzdHIsIFBhdGhdLAogICAgZGlzdHJpY3RfY29sOiBzdHIgPSAiRGlzdHJpY3QiLAogICAgbWFya2V0X2NvbDogc3RyID0gIk1hcmtldCIsCiAgICByYXdfbWFya2V0X2NvbDogc3RyID0gIk1hcmtldF9SYXciCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiCiAgICBBcHBsaWVzIGV2aWRlbmNlLWJhc2VkIG1hcmtldCBjYW5vbmljYWxpemF0aW9uIHRvIGEgRGF0YUZyYW1lOgogICAgMS4gUHJlc2VydmVzIG9yaWdpbmFsIG1hcmtldCBuYW1lIGluIGByYXdfbWFya2V0X2NvbGAgKGUuZy4gJ01hcmtldF9SYXcnKS4KICAgIDIuIE1hcHMgYG1hcmtldF9jb2xgIHRvIGBtYXJrZXRfY2Fub25pY2FsYCB1c2luZyB0aGUgYWNjZXB0ZWQgYWxpYXMgbWFwcGluZy4KICAgIDMuIElmIG5vIGFjY2VwdGVkIGFsaWFzIGlzIGZvdW5kLCBsZWF2ZXMgYG1hcmtldF9jb2xgIGlkZW50aWNhbCB0byBgcmF3X21hcmtldF9jb2xgLgogICAgCiAgICBQdXJlIGZ1bmN0aW9uOiByZXR1cm5zIGEgbmV3IERhdGFGcmFtZSBjb3B5LgogICAgIiIiCiAgICBvdXRfZGYgPSBkZi5jb3B5KCkKICAgIAogICAgIyAxLiBQcmVzZXJ2ZSBvcmlnaW5hbCBuYW1lCiAgICBpZiByYXdfbWFya2V0X2NvbCBub3QgaW4gb3V0X2RmLmNvbHVtbnM6CiAgICAgICAgb3V0X2RmW3Jhd19tYXJrZXRfY29sXSA9IG91dF9kZlttYXJrZXRfY29sXQogICAgICAgIAogICAgIyAyLiBSZXNvbHZlIG1hcHBpbmcKICAgIGlmIGlzaW5zdGFuY2UoYWxpYXNfc291cmNlLCBkaWN0KToKICAgICAgICBtYXBwaW5nID0gYWxpYXNfc291cmNlCiAgICBlbGlmIGlzaW5zdGFuY2UoYWxpYXNfc291cmNlLCBwZC5EYXRhRnJhbWUpOgogICAgICAgIG1hcHBpbmcgPSBidWlsZF9hbGlhc19tYXBwaW5nKGFsaWFzX3NvdXJjZSkKICAgIGVsc2U6CiAgICAgICAgYWxpYXNfZGYgPSBsb2FkX21hcmtldF9hbGlhc2VzKGFsaWFzX3NvdXJjZSkKICAgICAgICBtYXBwaW5nID0gYnVpbGRfYWxpYXNfbWFwcGluZyhhbGlhc19kZikKICAgICAgICAKICAgICMgMy4gQ2Fub25pY2FsaXplIE1hcmtldCBjb2x1bW4KICAgIGRlZiBfbWFwX3Jvdyhyb3cpOgogICAgICAgIGRpc3QgPSBzdHIocm93W2Rpc3RyaWN0X2NvbF0pLnN0cmlwKCkubG93ZXIoKQogICAgICAgIHJhd19ta3QgPSBzdHIocm93W3Jhd19tYXJrZXRfY29sXSkuc3RyaXAoKS5sb3dlcigpCiAgICAgICAgcmV0dXJuIG1hcHBpbmcuZ2V0KChkaXN0LCByYXdfbWt0KSwgcm93W3Jhd19tYXJrZXRfY29sXSkKICAgICAgICAKICAgIG91dF9kZlttYXJrZXRfY29sXSA9IG91dF9kZi5hcHBseShfbWFwX3JvdywgYXhpcz0xKQogICAgcmV0dXJuIG91dF9kZgoK",
    src_dir / "data_pipeline.py": "...",
    src_dir / "ingest_agmarknet_arrivals.py": "...",
    ref_dir / "market_aliases.csv": "ZGlzdHJpY3QsbWFya2V0X3JhdyxtYXJrZXRfY2Fub25pY2FsLHJ1bGUsZXZpZGVuY2UsZmlyc3RfZGF0ZSxsYXN0X2RhdGUsb3ZlcmxhcF9kYXlzLHByaWNlX2FncmVlbWVudCxzdGF0dXMKR2hhemlhYmFkLEdoYXppYWJhZCxHaGF6aWFiYWQsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxHaGF6aWFiYWQgQVBNQyxHaGF6aWFiYWQsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApHaGF6aWFiYWQsSGFwdXIsSGFwdXIsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxIYXB1ciBBUE1DLEhhcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0wIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTIwLDIwMjYtMDktMzAsMCwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIsTXVyYWRuYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIgQVBNQyxNdXJhZG5hZ2FyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0wIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTIwLDIwMjYtMDktMzAsMCwsQUNDRVBURUQKR2hhemlhYmFkLE5vaWRhLE5vaWRhLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgbWFya2V0IGJhc2VsaW5lLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApHaGF6aWFiYWQsTm9pZGEgQVBNQyxOb2lkYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0yMCwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkpoYW5zaSxCYXJ1d2FzYWdhcixCYXJ1d2FzYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKSmhhbnNpLEJhcnV3YXNhZ2FyIEFQTUMsQmFydXdhc2FnYXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApKaGFuc2ksQ2hpcmdhb24sQ2hpcmdhb24sSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxDaGlyZ2FvbiBBUE1DLENoaXJnYW9uLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0wIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTIwLDIwMjYtMDktMzAsMCwsQUNDRVBURUQKSmhhbnNpLEd1cnVzYXJhaSxHdXJ1c2FyYWksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxHdXJ1c2FyYWkgQVBNQyxHdXJ1c2FyYWksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpLEpoYW5zaSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSBBUE1DLEpoYW5zaSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0yMCwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkpoYW5zaSxKaGFuc2kgKEdyYWluKSxKaGFuc2kgKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSAoR3JhaW4pIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpIEdyYWluIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1cixNYXVyYW5pcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgbWFya2V0IGJhc2VsaW5lLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1ciBBUE1DLE1hdXJhbmlwdXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApKaGFuc2ksTW90aCxNb3RoLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgbWFya2V0IGJhc2VsaW5lLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTW90aCBBUE1DLE1vdGgsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApLYW5wdXIsQ2hvdWJlcHVyLENob3ViZXB1cixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLENob3ViZXB1ciBBUE1DLENob3ViZXB1ciwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0yMCwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkthbnB1cixLYW5wdXIoR3JhaW4pLEthbnB1cihHcmFpbiksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsS2FucHVyKEdyYWluKSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0yMCwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkthbnB1cixVdHRhcmlwdXJhLFV0dGFyaXB1cmEsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkthbnB1cixVdHRhcmlwdXJhIEFQTUMsVXR0YXJpcHVyYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0yMCwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkthbnB1cixWYXJpcGFhbCxWYXJpcGFhbCxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLFZhcmlwYWFsIEFQTUMsVmFyaXBhYWwsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApLYW5wdXIsQmFyaXBhYWwgQVBNQyxWYXJpcGFhbCxSZXZpZXcgUGVuZGluZyAocGhvbmV0aWMgdmFyaWFudCBiL3YpLE5vcm1hbGl6ZWQgbmFtZXMgZGlmZmVyIChiYXJpcGFhbCB2cyB2YXJpcGFhbCk7IHBlbmRpbmcgZXZpZGVuY2UgYWRqdWRpY2F0aW9uLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxVTk1BUFBFRC9SRVZJRVcKTHVja25vdyxCYW50aGFyYSxCYW50aGFyYSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKTHVja25vdyxCYW50aGFyYSBBUE1DLEJhbnRoYXJhLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0wIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTIwLDIwMjYtMDktMzAsMCwsQUNDRVBURUQKTHVja25vdyxMdWNrbm93LEx1Y2tub3csSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBtYXJrZXQgYmFzZWxpbmUsMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkx1Y2tub3csTHVja25vdyBBUE1DLEx1Y2tub3csIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApVbm5hbyxCYW5nYXJtYXUsQmFuZ2FybWF1LElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgbWFya2V0IGJhc2VsaW5lLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApVbm5hbyxCYW5nYXJtYXUgQVBNQyxCYW5nYXJtYXUsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMjAsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApVbm5hbyxQdXJ3YSxQdXJ3YSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIG1hcmtldCBiYXNlbGluZSwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sUHVyd2EgQVBNQyxQdXJ3YSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0yMCwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVEClVubmFvLFVubmFvLFVubmFvLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgbWFya2V0IGJhc2VsaW5lLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApVbm5hbyxVbm5hbyBBUE1DLFVubmFvLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0wIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTIwLDIwMjYtMDktMzAsMCwsQUNDRVBURUQK",
    out005_dir / "diff_77_deep_analysis.csv": "RGlzdHJpY3RfU3RkLE1hcmtldCxDb21tb2RpdHlfU3RkLFZhcmlldHksR3JhZGUsQXJyaXZhbF9EYXRlLE1vZGFsX1ByaWNlX0NsZWFuX2V4Y2VsLE1vZGFsX1ByaWNlX0NsZWFuX3BhbmVsLHByaWNlX2RpZmYscmF0aW8sZXhjZWxfaGFzX2RlY2ltYWxzLHBhbmVsX2hhc19kZWNpbWFscyxQcmljZV9Vbml0LEFycml2YWxfVW5pdF9Tb3VyY2UKR2hhemlhYmFkLE5vaWRhIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDMtMjUsMTc0Ni40NiwxNzQ1LjU5LDAuODcwMDAwMDAwMDAwMTE4MiwxLjAwMDQ5ODM5ODgyMjE3NDcsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxPbmlvbixPdGhlcixGQVEsMjAyNi0wMy0xNiwxNjIwLjQsMTYwMC4wLDIwLjQwMDAwMDAwMDAwMDA5LDEuMDEyNzUsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDMtMTIsMTczMy41NCwxNzMxLjA3LDIuNDcwMDAwMDAwMDAwMDI3MywxLjAwMTQyNjg2MzE1NDAwMyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLE9uaW9uLE90aGVyLEZBUSwyMDI2LTAyLTI3LDE2NTIuNzQsMTY3OS44MSwtMjcuMDY5OTk5OTk5OTk5OTM2LDAuOTgzODg1MDgyMjQxNDQ0LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDItMjUsMTgwNC44MywxOTE1LjU1LC0xMTAuNzIwMDAwMDAwMDAwMDMsMC45NDIxOTkzNjgzMjc2MzQzLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxPbmlvbixPdGhlcixGQVEsMjAyNi0wMy0yNSwxMDQ0LjI1LDEwMDAuMCw0NC4yNSwxLjA0NDI1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDMtMTIsMTAyMi4yLDEwMDAuMCwyMi4yMDAwMDAwMDAwMDAwNDUsMS4wMjIyLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDItMjcsMTIyNC42NywxMjE0Ljc4LDkuODkwMDAwMDAwMDAwMSwxLjAwODE0MTM5MTg1Njk2MixUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csTHVja25vdyBBUE1DLE9uaW9uLE90aGVyLEZBUSwyMDI2LTAzLTI1LDEwNzcuODEsMTA3Ny4wMywwLjc3OTk5OTk5OTk5OTk3MjcsMS4wMDA3MjQyMTM4MTAyMDAyLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxMdWNrbm93IEFQTUMsT25pb24sT3RoZXIsRkFRLDIwMjYtMDItMjcsMTIwNC45NSwxMjAwLjAsNC45NTAwMDAwMDAwMDAwNDU1LDEuMDA0MTI1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxNdXJhZG5hZ2FyIEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAyLTI3LDIwNDUuMjksMjAwMC4wLDQ1LjI4OTk5OTk5OTk5OTk2NCwxLjAyMjY0NSxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxUb21hdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjYsMTYzNC43NCwxNjA1LjEzLDI5LjYwOTk5OTk5OTk5OTksMS4wMTg0NDcxMDM5NzI4ODY4LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAzLTI1LDE4ODMuNjgsMTg4MC44NCwyLjg0MDAwMDAwMDAwMDE0NTUsMS4wMDE1MDk5NjM2MzMyNzAzLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAzLTEyLDIwMDAuNzUsMTYxOS41MiwzODEuMjMsMS4yMzUzOTY5MDc3MjU3NDU4LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxMdWNrbm93IEFQTUMsVG9tYXRvLE90aGVyLEZBUSwyMDI2LTAyLTI3LDExMDcuNzQsMTE3OS4yNCwtNzEuNSwwLjkzOTM2NzcyODM2NzQyMzEsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpKaGFuc2ksQ2hpcmdhb24gQVBNQyxXaGVhdCxPdGhlcixGQVEsMjAyNi0wMi0yNywyNDI2LjU1LDI0MjUuMCwxLjU1MDAwMDAwMDAwMDE4MiwxLjAwMDYzOTE3NTI1NzczMixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpKaGFuc2ksSmhhbnNpIEFQTUMsV2hlYXQsT3RoZXIsRkFRLDIwMjYtMDItMjcsMjUwMC4yMiwyNTAwLjQsLTAuMTgwMDAwMDAwMDAwMjkxMDQsMC45OTk5MjgwMTE1MTgxNTcsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpKaGFuc2ksTWF1cmFuaXB1ciBBUE1DLFdoZWF0LE90aGVyLEZBUSwyMDI2LTAyLTI3LDIzNTkuNjksMjQ1MC4wLC05MC4zMDk5OTk5OTk5OTk5NSwwLjk2MzEzODc3NTUxMDIwNDEsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTI3LDI0NTMuOCwyNDQxLjk3LDExLjgzMDAwMDAwMDAwMDM4MiwxLjAwNDg0NDQ0OTM1ODUwOTgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpLYW5wdXIsS2FucHVyKEdyYWluKSBBUE1DLFdoZWF0LE1lZGl1bSxGQVEsMjAyNi0wMi0yNywyNDY2LjY0LDIzODAuMCw4Ni42Mzk5OTk5OTk5OTk4NywxLjAzNjQwMzM2MTM0NDUzNzgsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFdoZWF0LERhcmEsRkFRLDIwMjYtMDItMjgsMjU2OC4wNiwyNDU4LjAsMTEwLjA1OTk5OTk5OTk5OTk1LDEuMDQ0Nzc2MjQwODQ2MjE2NSxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yNywyNDY3LjMsMjQyNS45LDQxLjQwMDAwMDAwMDAwMDA5LDEuMDE3MDY1ODMxMjM3ODkxLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFdoZWF0LERhcmEsRkFRLDIwMjYtMDItMjYsMjM2MC42MiwyNDQ2LjAsLTg1LjM4MDAwMDAwMDAwMDExLDAuOTY1MDk0MDMxMDcxMTM2NixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yNSwyNDU1LjkyLDI1MjYuMCwtNzAuMDc5OTk5OTk5OTk5OTMsMC45NzIyNTY1MzIwNjY1MDgzLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTI0LDI1MzcuNzcsMjQxNC4wLDEyMy43Njk5OTk5OTk5OTk5OCwxLjA1MTI3MTc0ODEzNTg3NCxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yMywyNDEyLjM0LDI0NjQuMCwtNTEuNjU5OTk5OTk5OTk5ODU0LDAuOTc5MDM0MDkwOTA5MDkxLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTIyLDI0NjguMzcsMjQ3NC4wLC01LjYzMDAwMDAwMDAwMDEwOSwwLjk5NzcyNDMzMzA2Mzg2NDEsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKVW5uYW8sQmFuZ2FybWF1IEFQTUMsV2hlYXQsRGFyYSxGQVEsMjAyNi0wMi0yNywyMjUwLjEsMjI1MC4xOCwtMC4wNzk5OTk5OTk5OTk5MjcyNCwwLjk5OTk2NDQ0NzI4ODY2MTQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpVbm5hbyxCYW5nYXJtYXUgQVBNQyxXaGVhdCxEYXJhLEZBUSwyMDI2LTAyLTI1LDIyOTAuOTksMjI4NS4zMSw1LjY3OTk5OTk5OTk5OTgzNiwxLjAwMjQ4NTQzOTYxMjEzMTQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsR2hhemlhYmFkIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMjksNTQzOS4wOCw1MzQxLjc4LDk3LjMwMDAwMDAwMDAwMDE4LDEuMDE4MjE0OTAyMTQ4NzIyLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEdoYXppYWJhZCBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMjQsMTE5MDQuNjcsNzU2Mi4yNSw0MzQyLjQyLDEuNTc0MjIzMjgwMTA4NDMzMyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMzAsMzA4My45LDExNzE4LjAsLTg2MzQuMSwwLjI2MzE3NjMwOTk1MDUwMzUsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsUmljZSxPdGhlcixGQVEsMjAyNi0wMy0xMyw3NTU4LjA2LDUzMDAuMCwyMjU4LjA2MDAwMDAwMDAwMDQsMS40MjYwNDkwNTY2MDM3NzM2LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMTIsNjA2MC4xNSw0ODU5LjUyLDEyMDAuNjI5OTk5OTk5OTk5MiwxLjI0NzA2NzYxMTYxNTk2MTgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpLYW5wdXIsS2FucHVyKEdyYWluKSBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDMtMTYsNDMwMy42MywzNjM3LjU4LDY2Ni4wNTAwMDAwMDAwMDAyLDEuMTgzMTAyNTAyMjEzMDA5NyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkthbnB1cixLYW5wdXIoR3JhaW4pIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDItMjcsNDM1NC4xMiwyNzM1LjAsMTYxOS4xMiwxLjU5MTk5OTk5OTk5OTk5OTksVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFJpY2UsQ29tbW9uLEZBUSwyMDI2LTAzLTI5LDI1MzkuNjgsMjIxNy4yOSwzMjIuMzg5OTk5OTk5OTk5OSwxLjE0NTM5ODIxMTMzMDA0NyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0yOCwyMjA1LjQxLDIwMDEuNywyMDMuNzA5OTk5OTk5OTk5OCwxLjEwMTc2ODQ5Njc3NzczODgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMjcsMjc2OC4zOSwyMjAxLjUyLDU2Ni44Njk5OTk5OTk5OTk5LDEuMjU3NDkwMjc5NDQzMjkzNixUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0yNCw0ODY0LjY4LDIwMDAuMCwyODY0LjY4MDAwMDAwMDAwMDMsMi40MzIzNCxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMTgsNjM5MC42OCwyODA1LjAsMzU4NS42ODAwMDAwMDAwMDAzLDIuMjc4MzE3MjkwNTUyNTg0NixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEJhbnRoYXJhIEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMTYsNTgxOC42NywzNzQyLjAsMjA3Ni42NywxLjU1NDk2MjU4Njg1MTk1MDgsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFJpY2UsQ29tbW9uLEZBUSwyMDI2LTAzLTA3LDI0NzkuMjgsMjQxNS4zLDYzLjk4MDAwMDAwMDAwMDAyLDEuMDI2NDg5NDYzMDA2NjY1OSxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLE90aGVyLEZBUSwyMDI2LTAzLTAyLDU2MjYuNjgsMjQwMC4wLDMyMjYuNjgwMDAwMDAwMDAwMywyLjM0NDQ1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csQmFudGhhcmEgQVBNQyxSaWNlLE90aGVyLEZBUSwyMDI2LTAyLTI4LDU5MzEuOTgsMjMyNy4wLDM2MDQuOTc5OTk5OTk5OTk5NiwyLjU0OTE5NjM5MDIwMTk3NjcsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxCYW50aGFyYSBBUE1DLFJpY2UsQ29tbW9uLEZBUSwyMDI2LTAyLTI1LDMyNTAuMTgsMjQwMC4wLDg1MC4xNzk5OTk5OTk5OTk4LDEuMzU0MjQxNjY2NjY2NjY2NixUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0yNyw0NzExLjMxLDQyMjcuMTEsNDg0LjIwMDAwMDAwMDAwMDcsMS4xMTQ1NDYzNDQ5MDIzMDk0LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKTHVja25vdyxMdWNrbm93IEFQTUMsUmljZSxDb21tb24sRkFRLDIwMjYtMDMtMjAsNDcxNS45NSw0NjA5Ljg2LDEwNi4wOTAwMDAwMDAwMDAxNSwxLjAyMzAxMzcxNDA4MjQyMzQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0xOCw2MTE0LjEzLDM3MDcuMjUsMjQwNi44OCwxLjY0OTIzNTk1NjU3MTU4MjgsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMy0wNiw0NjEzLjM3LDM5MTYuMzIsNjk3LjA0OTk5OTk5OTk5OTcsMS4xNzc5ODU5NjY0MTc0NTMsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpMdWNrbm93LEx1Y2tub3cgQVBNQyxSaWNlLENvbW1vbixGQVEsMjAyNi0wMi0yNyw0ODgzLjUxLDQ3MzQuNjYsMTQ4Ljg1MDAwMDAwMDAwMDM2LDEuMDMxNDM4MzcxNDk4NjkyOCxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csTHVja25vdyBBUE1DLFJpY2UsT3RoZXIsRkFRLDIwMjYtMDItMjcsNzA5OC43Nyw1MTAwLjAsMTk5OC43NzAwMDAwMDAwMDA0LDEuMzkxOTE1Njg2Mjc0NTEsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEdoYXppYWJhZCBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNywxMTAxLjc4LDExMTAuMDIsLTguMjQwMDAwMDAwMDAwMDA5LDAuOTkyNTc2NzEwMzI5NTQzNixUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0zMSw2MTEuNDgsMTEwMC4wLC00ODguNTIsMC41NTU4OTA5MDkwOTA5MDkxLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0zMCw4MDQuNjMsOTQ3LjQ1LC0xNDIuODIwMDAwMDAwMDAwMDUsMC44NDkyNTg1MzYwNzA1MDQ5LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI5LDk1OC4wOCwxMDcwLjU3LC0xMTIuNDg5OTk5OTk5OTk5OSwwLjg5NDkyNTEzMzM0MDE4MzMsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsSGFwdXIgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjgsOTE1Ljc3LDk3Ni4zNywtNjAuNjAwMDAwMDAwMDAwMDIsMC45Mzc5MzMzNjU0MjQ5OTI1LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI3LDk3Ny40MywxMDEwLjgyLC0zMy4zOTAwMDAwMDAwMDAxLDAuOTY2OTY3NDEyNTk1NzE0MyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yNiw5ODUuODcsMTA5MS42OSwtMTA1LjgyMDAwMDAwMDAwMDA1LDAuOTAzMDY3NzIwNjg5OTM5NSxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yNSw4NjMuMTgsOTQxLjU2LC03OC4zOCwwLjkxNjc1NTE3MjI2NzMwMTEsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsSGFwdXIgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjQsOTEwLjAxLDk0OS40MSwtMzkuMzk5OTk5OTk5OTk5OTgsMC45NTg1MDA1NDI0NDIxNDgzLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTIzLDk3MC41LDEwMzYuNjQsLTY2LjE0MDAwMDAwMDAwMDEsMC45MzYxOTc3MTU2OTY4NjY3LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTIyLDgyMi4zLDEwNjEuOTUsLTIzOS42NTAwMDAwMDAwMDAxLDAuNzc0MzMwMjQxNTM2Nzk1NCxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxIYXB1ciBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yMCw4MDYuNTgsOTI0LjI0LC0xMTcuNjU5OTk5OTk5OTk5OTcsMC44NzI2OTU0MDM3OTEyMjMxLFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTAzLDkzMC4yOSw5NzguNDgsLTQ4LjE5MDAwMDAwMDAwMDA1NSwwLjk1MDc1MDE0MzA3OTA2MTQsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsSGFwdXIgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMDEsMTA0Mi4yNCwxMDUyLjk3LC0xMC43MzAwMDAwMDAwMDAwMTgsMC45ODk4MDk3NzYxNTY5NjU1LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLEhhcHVyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAyLTI3LDg3NS44OCw4MzYuMzIsMzkuNTU5OTk5OTk5OTk5OTQ1LDEuMDQ3MzAyNDY3OTU0ODQ5NyxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxNdXJhZG5hZ2FyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTMxLDg2OC42OCw4MDAuMCw2OC42Nzk5OTk5OTk5OTk5NSwxLjA4NTg1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxNdXJhZG5hZ2FyIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI2LDg1OS4xLDgwMC4wLDU5LjEwMDAwMDAwMDAwMDAyLDEuMDczODc1MDAwMDAwMDAwMSxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMzAsMTU5Ni4yMyw3MDAuMCw4OTYuMjMsMi4yODAzMjg1NzE0Mjg1NzE1LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMy0yOSw5OTMuNjYsNzAwLjAsMjkzLjY1OTk5OTk5OTk5OTk3LDEuNDE5NTE0Mjg1NzE0Mjg1NyxUcnVlLEZhbHNlLFJzLi9RdWludGFsLFRvbm5lcwpHaGF6aWFiYWQsTm9pZGEgQVBNQyxQb3RhdG8sT3RoZXIsRkFRLDIwMjYtMDMtMjYsODMzLjIyLDcwMC4wLDEzMy4yMjAwMDAwMDAwMDAwMywxLjE5MDMxNDI4NTcxNDI4NTcsVHJ1ZSxGYWxzZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTI1LDg1OS4yOCw4NTYuMjIsMy4wNTk5OTk5OTk5OTk5NDU0LDEuMDAzNTczODQ3ODQyODQ0LFRydWUsVHJ1ZSxScy4vUXVpbnRhbCxUb25uZXMKR2hhemlhYmFkLE5vaWRhIEFQTUMsUG90YXRvLE90aGVyLEZBUSwyMDI2LTAzLTEyLDk5My40Nyw5MzMuMDksNjAuMzc5OTk5OTk5OTk5OTk1LDEuMDY0NzA5NzI3ODkzMzQzNSxUcnVlLFRydWUsUnMuL1F1aW50YWwsVG9ubmVzCkdoYXppYWJhZCxOb2lkYSBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNyw5MDYuNzMsOTEyLjY4LC01Ljk0OTk5OTk5OTk5OTkzMiwwLjk5MzQ4MDczODA0NjE5MzcsVHJ1ZSxUcnVlLFJzLi9RdWludGFsLFRvbm5lcwpLYW5wdXIsS2FucHVyKEdyYWluKSBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNyw2MDEuMjQsNjAwLjAsMS4yNDAwMDAwMDAwMDAwMDksMS4wMDIwNjY2NjY2NjY2NjY3LFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCkx1Y2tub3csTHVja25vdyBBUE1DLFBvdGF0byxPdGhlcixGQVEsMjAyNi0wMi0yNyw3MDEuMDgsNzAwLjAsMS4wODAwMDAwMDAwMDAwNDEsMS4wMDE1NDI4NTcxNDI4NTczLFRydWUsRmFsc2UsUnMuL1F1aW50YWwsVG9ubmVzCg=="
}

print("\n=== BOOTSTRAP SOURCE MODULES & REFERENCE DATA ===")
for dest_path, b64_str in b64_payloads.items():
    dest_path.write_bytes(base64.b64decode(b64_str))
    print(f"Bootstrapped: {dest_path} ({dest_path.stat().st_size:,} bytes)")

# Resolve dataset root in Kaggle environment
KAGGLE_INPUT_DIR = "/kaggle/input/mandi-prediction-audit-003b"
if not os.path.exists(KAGGLE_INPUT_DIR):
    KAGGLE_INPUT_DIR = "/kaggle/input/datasets/theankitchoubey/mandi-prediction-audit-003b"
if not os.path.exists(KAGGLE_INPUT_DIR):
    cands = glob.glob("/kaggle/input/**/mandi_5year_validated.csv", recursive=True)
    if cands:
        KAGGLE_INPUT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(cands[0])))
    else:
        KAGGLE_INPUT_DIR = "."

print(f"\nResolved Dataset Payload Root: {KAGGLE_INPUT_DIR}")

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

input_files = [
    os.path.join(KAGGLE_INPUT_DIR, "data/processed/mandi_5year_validated.csv"),
    os.path.join(KAGGLE_INPUT_DIR, "data/processed/validation_summary.json"),
    os.path.join(KAGGLE_INPUT_DIR, "data/processed/processing_manifest.json")
]
input_files.extend(sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "5 year dataset/*.csv"))))
input_files.extend(sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/*.xlsx"))))
if not glob.glob(os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/*.xlsx")):
    input_files.extend(sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/raw_downloads/*.xlsx"))))

ceda_matches = glob.glob(f"{KAGGLE_INPUT_DIR}/**/ceda price data.csv", recursive=True)
if ceda_matches:
    input_files.append(ceda_matches[0])

print("\n=== INPUT FILES & SHA-256 MANIFEST ===")
for ipath in input_files:
    if os.path.exists(ipath):
        sz = os.path.getsize(ipath)
        sha = compute_sha256(ipath)
        print(f"INPUT: {os.path.basename(ipath):<35} | {sz:>10,} bytes | sha256: {sha}")


In [ ]:
# Cell 2: Part 1 - Market Inventory per District & Evidence Rule Evaluation
from src.market_identity import normalize_market_name, evaluate_alias_evidence

raw_csv_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "5 year dataset/*.csv")))
if not raw_csv_files:
    raw_csv_files = sorted(glob.glob("5 year dataset/*.csv"))
print(f"Found {len(raw_csv_files)} raw CSV files.")

raw_dfs = []
for f in raw_csv_files:
    df_f = pd.read_csv(f)
    raw_dfs.append(df_f)
df_raw_all = pd.concat(raw_dfs, ignore_index=True)
print(f"Total raw observations loaded: {len(df_raw_all):,}")

# Parse arrival dates
df_raw_all["Arrival_Date_dt"] = pd.to_datetime(df_raw_all["Arrival_Date"], format="%d/%m/%Y", errors="coerce")
DISTRICT_NORMALIZATION = {
    "gaziabaad": "Ghaziabad",
    "ghaziabad": "Ghaziabad",
    "gaziabad": "Ghaziabad",
    "jhansi": "Jhansi",
    "kanpur": "Kanpur",
    "kanpur nagar": "Kanpur",
    "lukhnow": "Lucknow",
    "lucknow": "Lucknow",
    "unnao": "Unnao"
}
df_raw_all["District_Norm"] = df_raw_all["District"].astype(str).str.strip().str.lower().map(DISTRICT_NORMALIZATION).fillna(df_raw_all["District"].astype(str).str.strip())
df_raw_all["Market_Raw"] = df_raw_all["Market"].astype(str).str.strip()

# 1. Market Inventory per District
inventory_records = []
for (dist, mkt), grp in df_raw_all.groupby(["District_Norm", "Market_Raw"]):
    dts = grp["Arrival_Date_dt"].dropna().sort_values()
    first_dt = dts.iloc[0] if len(dts) > 0 else pd.NaT
    last_dt = dts.iloc[-1] if len(dts) > 0 else pd.NaT
    inventory_records.append({
        "district": dist,
        "market_raw": mkt,
        "normalized_token": normalize_market_name(mkt),
        "row_count": len(grp),
        "commodity_count": grp["Commodity"].nunique(),
        "first_date": first_dt.strftime("%Y-%m-%d") if pd.notna(first_dt) else "",
        "last_date": last_dt.strftime("%Y-%m-%d") if pd.notna(last_dt) else "",
        "has_apmc_suffix": bool(mkt.endswith(" APMC") or mkt.endswith("APMC"))
    })

df_inventory = pd.DataFrame(inventory_records).sort_values(["district", "market_raw"]).reset_index(drop=True)
df_inventory.to_csv("market_inventory_raw.csv", index=False)
print(f"Exported market_inventory_raw.csv ({len(df_inventory)} distinct markets across districts)")

# 2. Candidate Alias Pairs and Evidence Rule Evaluation
candidate_alias_rows = []
accepted_alias_rows = []
review_alias_rows = []

for dist, dist_grp in df_inventory.groupby("district"):
    for token, token_grp in dist_grp.groupby("normalized_token"):
        if len(token_grp) == 1:
            row = token_grp.iloc[0]
            accepted_alias_rows.append({
                "district": dist,
                "market_raw": row["market_raw"],
                "market_canonical": row["market_raw"],
                "rule": "Identity (single token)",
                "evidence": "Unique market for normalized token in district",
                "first_date": row["first_date"],
                "last_date": row["last_date"],
                "overlap_days": 0,
                "price_agreement": None,
                "status": "ACCEPTED"
            })
        else:
            non_apmc = token_grp[~token_grp["has_apmc_suffix"]]
            if not non_apmc.empty:
                canonical_name = non_apmc.sort_values("first_date").iloc[0]["market_raw"]
            else:
                canonical_name = token_grp.sort_values("first_date").iloc[0]["market_raw"]
                
            anchor_info = token_grp[token_grp["market_raw"] == canonical_name].iloc[0]
            
            accepted_alias_rows.append({
                "district": dist,
                "market_raw": canonical_name,
                "market_canonical": canonical_name,
                "rule": "Identity (canonical anchor)",
                "evidence": "Canonical baseline anchor for normalized token",
                "first_date": anchor_info["first_date"],
                "last_date": anchor_info["last_date"],
                "overlap_days": 0,
                "price_agreement": None,
                "status": "ACCEPTED"
            })
            
            for _, other_row in token_grp[token_grp["market_raw"] != canonical_name].iterrows():
                other_raw = other_row["market_raw"]
                
                data_anchor = df_raw_all[(df_raw_all["District_Norm"] == dist) & (df_raw_all["Market_Raw"] == canonical_name)]
                data_other = df_raw_all[(df_raw_all["District_Norm"] == dist) & (df_raw_all["Market_Raw"] == other_raw)]
                
                merge_key = ["Commodity", "Variety", "Grade", "Arrival_Date_dt"]
                overlap_match = pd.merge(
                    data_anchor[merge_key + ["Modal_Price"]],
                    data_other[merge_key + ["Modal_Price"]],
                    on=merge_key,
                    suffixes=("_anchor", "_other")
                )
                overlap_match.rename(columns={"Modal_Price_anchor": "price_1", "Modal_Price_other": "price_2"}, inplace=True)
                
                is_accepted, ev_dict = evaluate_alias_evidence(
                    district_1=dist,
                    name_1=canonical_name,
                    first_date_1=anchor_info["first_date"],
                    last_date_1=anchor_info["last_date"],
                    district_2=dist,
                    name_2=other_raw,
                    first_date_2=other_row["first_date"],
                    last_date_2=other_row["last_date"],
                    overlapping_records_df=overlap_match
                )
                
                cand_record = {
                    "district": dist,
                    "market_raw": other_raw,
                    "market_canonical": canonical_name,
                    "rule": ev_dict["rule"],
                    "evidence": ev_dict["evidence"],
                    "first_date": other_row["first_date"],
                    "last_date": other_row["last_date"],
                    "overlap_days": ev_dict["overlap_days"],
                    "price_agreement": ev_dict["price_agreement"],
                    "status": ev_dict["status"]
                }
                candidate_alias_rows.append(cand_record)
                
                if is_accepted:
                    accepted_alias_rows.append(cand_record)
                else:
                    review_alias_rows.append(cand_record)

# Cross-phonetic inspection: Baripaal APMC vs Varipaal in Kanpur
kanpur_baripaal = df_inventory[(df_inventory["district"] == "Kanpur") & (df_inventory["market_raw"] == "Baripaal APMC")]
if not kanpur_baripaal.empty:
    b_row = kanpur_baripaal.iloc[0]
    review_alias_rows.append({
        "district": "Kanpur",
        "market_raw": "Baripaal APMC",
        "market_canonical": "Varipaal",
        "rule": "Review Pending (cross-phonetic b/v)",
        "evidence": "Normalized mismatch ('baripaal' vs 'varipaal'); candidate for manual verification",
        "first_date": b_row["first_date"],
        "last_date": b_row["last_date"],
        "overlap_days": 0,
        "price_agreement": None,
        "status": "UNMAPPED/REVIEW"
    })

df_alias_candidates = pd.DataFrame(candidate_alias_rows)
df_alias_accepted = pd.DataFrame(accepted_alias_rows).sort_values(["district", "market_canonical", "market_raw"]).reset_index(drop=True)
df_alias_review = pd.DataFrame(review_alias_rows)

df_alias_candidates.to_csv("market_alias_candidates.csv", index=False)
df_alias_accepted.to_csv("market_aliases_accepted.csv", index=False)
df_alias_review.to_csv("market_aliases_review.csv", index=False)

print("\n=== EVIDENCE-BASED ALIAS EVALUATION SUMMARY ===")
print(f"Total Market Inventory Entries: {len(df_inventory)}")
print(f"Candidate Alias Pairs Evaluated: {len(df_alias_candidates)}")
print(f"Accepted Alias Table Entries: {len(df_alias_accepted)}")
print(f"Review / Unmapped Entries: {len(df_alias_review)}")


In [ ]:
# Cell 3: Part 2 - Pipeline v2 Execution on the 5-Year Raw Files & Comparison with v1
from src.data_pipeline import MandiDataPipeline

print("Executing MandiDataPipeline v2 on 5-year dataset...")
pipeline_v2 = MandiDataPipeline(
    input_dir=os.path.join(KAGGLE_INPUT_DIR, "5 year dataset"),
    output_dir=".",
    output_filename="mandi_5year_validated_v2.csv",
    manifest_filename="processing_manifest_v2.json",
    min_series_records=20,
    exclude_ambiguous_same_grade=True,
    pipeline_version="v2",
    alias_df=df_alias_accepted
)

df_v2, manifest_v2, out_csv_v2, out_manifest_v2 = pipeline_v2.run(raw_csv_files)
print(f"Pipeline v2 Complete! Accepted clean rows: {len(df_v2):,} across {df_v2['series_id'].nunique():,} unique series.")

# Load v1 panel for disposition comparison
val_csv_v1_path = os.path.join(KAGGLE_INPUT_DIR, "data/processed/mandi_5year_validated.csv")
df_v1 = pd.read_csv(val_csv_v1_path)
df_v1["Arrival_Date"] = pd.to_datetime(df_v1["Arrival_Date"]).dt.strftime("%Y-%m-%d")
print(f"Loaded v1 validated panel: {len(df_v1):,} rows across {df_v1['series_id'].nunique():,} series.")

# Detailed disposition accounting
with open(os.path.join(KAGGLE_INPUT_DIR, "data/processed/validation_summary.json")) as f:
    summary_v1 = json.load(f)

v1_counts = summary_v1["counts"]
v2_metrics = manifest_v2["summary_metrics"]

disposition_comparison = {
    "v1_pipeline": {
        "raw_rows": v1_counts["total_input_raw_records"],
        "invalid_dropped_rows": v1_counts["total_quarantined_invalid_records"],
        "conflicting_quarantined_rows": v1_counts["total_quarantined_conflicting_records"],
        "exact_duplicates_deduped": v1_counts["total_exact_duplicates_deduplicated"],
        "low_obs_dropped_rows": v1_counts["total_low_observation_series_dropped_rows"],
        "accepted_clean_rows": v1_counts["total_accepted_clean_records"],
        "unique_series_count": v1_counts["unique_series_retained"],
        "unique_markets_count": v1_counts["unique_markets_count"]
    },
    "v2_pipeline": {
        "raw_rows": v2_metrics["total_raw_rows"],
        "invalid_dropped_rows": v2_metrics["total_dropped_raw_rows"],
        "conflicting_quarantined_rows": v2_metrics["ambiguous_same_grade_excluded_rows"],
        "exact_duplicates_deduped": v2_metrics["exact_duplicates_deduplicated"],
        "low_obs_dropped_rows": v2_metrics["low_observation_series_dropped_rows"],
        "accepted_clean_rows": v2_metrics["final_clean_rows"],
        "unique_series_count": v2_metrics["unique_series_count"],
        "unique_markets_count": v2_metrics["unique_markets_count"]
    }
}

rescued_rows = disposition_comparison["v1_pipeline"]["low_obs_dropped_rows"] - disposition_comparison["v2_pipeline"]["low_obs_dropped_rows"]
net_accepted_gain = disposition_comparison["v2_pipeline"]["accepted_clean_rows"] - disposition_comparison["v1_pipeline"]["accepted_clean_rows"]

print("\n=== PIPELINE DISPOSITION COMPARISON (v1 vs v2) ===")
print(f"v1 Accepted: {disposition_comparison['v1_pipeline']['accepted_clean_rows']:,} | v2 Accepted: {disposition_comparison['v2_pipeline']['accepted_clean_rows']:,} (Net Gain: +{net_accepted_gain:,} rows)")
print(f"v1 Low-Obs Dropped: {disposition_comparison['v1_pipeline']['low_obs_dropped_rows']:,} | v2 Low-Obs Dropped: {disposition_comparison['v2_pipeline']['low_obs_dropped_rows']:,} (Rescued Rows: +{rescued_rows:,} rows)")
print(f"v1 Series: {disposition_comparison['v1_pipeline']['unique_series_count']:,} | v2 Series: {disposition_comparison['v2_pipeline']['unique_series_count']:,}")
print(f"v1 Markets: {disposition_comparison['v1_pipeline']['unique_markets_count']:,} | v2 Markets: {disposition_comparison['v2_pipeline']['unique_markets_count']:,}")

# Monthly comparison breakdown
df_v1["year_month"] = df_v1["Arrival_Date"].str[:7]
df_v2["year_month"] = pd.to_datetime(df_v2["Arrival_Date"]).dt.strftime("%Y-%m")

monthly_v1 = df_v1.groupby("year_month").size().reset_index(name="v1_accepted_rows")
monthly_v2 = df_v2.groupby("year_month").size().reset_index(name="v2_accepted_rows")
monthly_comp = pd.merge(monthly_v1, monthly_v2, on="year_month", how="outer").fillna(0)
monthly_comp["accepted_diff"] = monthly_comp["v2_accepted_rows"] - monthly_comp["v1_accepted_rows"]
monthly_comp.to_csv("panel_v1_vs_v2_comparison.csv", index=False)
print("Saved panel_v1_vs_v2_comparison.csv")

with open("pipeline_disposition_summary.json", "w") as f:
    json.dump({
        "comparison": disposition_comparison,
        "rescued_low_obs_rows": int(rescued_rows),
        "net_accepted_gain": int(net_accepted_gain)
    }, f, indent=2)
print("Saved pipeline_disposition_summary.json")


In [ ]:
# Cell 4: Part 3 - Excel and CEDA Corroboration on v2
from src.ingest_agmarknet_arrivals import load_arrivals_file
from src.market_identity import apply_market_canonicalization

alias_map = dict(zip(df_alias_accepted["market_raw"], df_alias_accepted["market_canonical"]))

excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(f"{KAGGLE_INPUT_DIR}/**/raw_downloads/*.xlsx", recursive=True))

excel_dfs = [load_arrivals_file(f) for f in excel_files]
df_excel = pd.concat(excel_dfs, ignore_index=True)
df_excel["Arrival_Date"] = pd.to_datetime(df_excel["Arrival_Date"]).dt.strftime("%Y-%m-%d")
df_excel["Modal_Price_Clean"] = pd.to_numeric(df_excel["Modal_Price"], errors="coerce")

# Filter in-window Excel records (<= 2026-09-30)
df_excel_in_window = df_excel[df_excel["Arrival_Date"] <= "2026-09-30"].copy()
print(f"Total Excel rows in 5 districts within panel window: {len(df_excel_in_window):,}")

# Canonicalize Market in df_excel using accepted alias table
df_excel_canon = apply_market_canonicalization(df_excel_in_window, df_alias_accepted)

df_v2_corrob = df_v2.copy()
df_v2_corrob["Arrival_Date"] = pd.to_datetime(df_v2_corrob["Arrival_Date"]).dt.strftime("%Y-%m-%d")
df_v2_corrob["Modal_Price_Clean"] = pd.to_numeric(df_v2_corrob["Modal_Price"], errors="coerce")

match_cols_excel = ["District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date"]
match_cols_panel = ["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"]

merged_excel_v2 = pd.merge(
    df_excel_canon,
    df_v2_corrob,
    left_on=match_cols_excel,
    right_on=match_cols_panel,
    how="inner",
    suffixes=("_excel", "_panel_v2")
)

merged_excel_v2["price_diff"] = merged_excel_v2["Modal_Price_Clean_excel"] - merged_excel_v2["Modal_Price_Clean_panel_v2"]
exact_matches_v2 = int((merged_excel_v2["price_diff"] == 0).sum())
diff_count_v2 = len(merged_excel_v2) - exact_matches_v2

print("\n=== AGMARKNET EXCEL CORROBORATION ON PANEL V2 ===")
print(f"In-Window Excel Records: {len(df_excel_in_window):,}")
print(f"Matched in Panel v2: {len(merged_excel_v2):,} ({len(merged_excel_v2)/len(df_excel_in_window)*100:.2f}%)")
print(f"Exact Modal Price Matches: {exact_matches_v2:,} ({exact_matches_v2/len(merged_excel_v2)*100:.2f}%)")
print(f"Non-Zero Price Differences: {diff_count_v2:,} ({diff_count_v2/len(merged_excel_v2)*100:.2f}%)")

# CEDA Corroboration
ceda_files = glob.glob(f"{KAGGLE_INPUT_DIR}/**/ceda price data.csv", recursive=True)
if ceda_files:
    df_ceda = pd.read_csv(ceda_files[0])
    df_ceda["District"] = df_ceda["district_name"].astype(str).str.strip().str.lower().map(DISTRICT_NORMALIZATION).fillna(df_ceda["district_name"].astype(str).str.strip())
    df_ceda["Market_Raw"] = df_ceda["market_name"].astype(str).str.strip()
    df_ceda["Market"] = df_ceda["Market_Raw"].map(alias_map).fillna(df_ceda["Market_Raw"])
    df_ceda["Commodity"] = df_ceda["cmdty"].astype(str).str.strip()
    df_ceda["Variety"] = df_ceda["variety"].astype(str).str.strip()
    df_ceda["Grade"] = "FAQ"
    df_ceda["Arrival_Date"] = pd.to_datetime(df_ceda["t"]).dt.strftime("%Y-%m-%d")
    df_ceda["Modal_Price_Clean"] = pd.to_numeric(df_ceda["p_modal"], errors="coerce")
    
    merged_ceda_v2 = pd.merge(
        df_ceda,
        df_v2_corrob,
        on=["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"],
        how="inner",
        suffixes=("_ceda", "_panel_v2")
    )
    merged_ceda_v2["price_diff"] = merged_ceda_v2["Modal_Price_Clean_ceda"] - merged_ceda_v2["Modal_Price_Clean_panel_v2"]
    ceda_exact = int((merged_ceda_v2["price_diff"] == 0).sum())
    print(f"\n=== CEDA CORROBORATION ON PANEL V2 ===")
    print(f"Total CEDA Records: {len(df_ceda):,}")
    print(f"Matched in Panel v2: {len(merged_ceda_v2):,} ({len(merged_ceda_v2)/len(df_ceda)*100:.2f}%)")
    print(f"Exact Matches: {ceda_exact:,} ({ceda_exact/len(merged_ceda_v2)*100:.2f}%)")
else:
    merged_ceda_v2 = pd.DataFrame()
    ceda_exact = 0

corrob_summary = {
    "excel_corroboration": {
        "in_window_excel_rows": len(df_excel_in_window),
        "matched_rows_v2": len(merged_excel_v2),
        "match_rate_pct": round(len(merged_excel_v2) / len(df_excel_in_window) * 100, 2),
        "exact_price_matches": exact_matches_v2,
        "exact_match_rate_pct": round(exact_matches_v2 / len(merged_excel_v2) * 100, 2),
        "non_zero_price_differences": diff_count_v2
    },
    "ceda_corroboration": {
        "total_ceda_rows": len(df_ceda) if ceda_files else 0,
        "matched_rows_v2": len(merged_ceda_v2),
        "exact_matches": ceda_exact
    }
}
with open("corroboration_v2_summary.json", "w") as f:
    json.dump(corrob_summary, f, indent=2)
print("Saved corroboration_v2_summary.json")


In [ ]:
# Cell 5: Part 4 - Regime Diagnostics (Monthly 2021-09 to 2026-09 on v2)
df_v2["year_month"] = pd.to_datetime(df_v2["Arrival_Date"]).dt.strftime("%Y-%m")
df_v2["Modal_Price_Clean"] = pd.to_numeric(df_v2["Modal_Price"], errors="coerce")
df_v2["Min_Price_Clean"] = pd.to_numeric(df_v2["Min_Price"], errors="coerce")
df_v2["Max_Price_Clean"] = pd.to_numeric(df_v2["Max_Price"], errors="coerce")

df_v2["has_apmc_suffix"] = df_v2["Market_Raw"].str.contains("APMC", case=False, na=False)
df_v2["is_fractional"] = (df_v2["Modal_Price_Clean"] % 1 != 0)
df_v2["is_min_max_inconsistent"] = (
    (df_v2["Min_Price_Clean"] > df_v2["Max_Price_Clean"]) |
    (df_v2["Modal_Price_Clean"] < df_v2["Min_Price_Clean"]) |
    (df_v2["Modal_Price_Clean"] > df_v2["Max_Price_Clean"])
).fillna(False)

regime_records = []
for ym, grp in df_v2.groupby("year_month"):
    n_obs = len(grp)
    n_suffix = int(grp["has_apmc_suffix"].sum())
    n_frac = int(grp["is_fractional"].sum())
    n_min_max = int(grp["is_min_max_inconsistent"].sum())
    
    # Active markets reporting per district
    mkt_counts_per_dist = grp.groupby("District")["Market"].nunique().to_dict()
    total_active_markets = grp["Market"].nunique()
    
    regime_records.append({
        "year_month": ym,
        "total_rows": n_obs,
        "active_markets_count": total_active_markets,
        "markets_reporting_by_district": mkt_counts_per_dist,
        "suffix_apmc_rows": n_suffix,
        "suffix_apmc_share": round(n_suffix / n_obs, 4),
        "fractional_price_rows": n_frac,
        "fractional_price_share": round(n_frac / n_obs, 4),
        "min_max_inconsistent_rows": n_min_max,
        "min_max_inconsistent_share": round(n_min_max / n_obs, 4),
        "is_post_agmarknet_2_0": (ym >= "2025-11"), # 2025-11-07 go-live
        "is_post_up_integration": (ym >= "2026-09")  # 2026-09-29 UP mandis finished
    })

df_regime = pd.DataFrame(regime_records)
df_regime_csv = df_regime.copy()
for dist in ["Ghaziabad", "Jhansi", "Kanpur", "Lucknow", "Unnao"]:
    df_regime_csv[f"active_markets_{dist.lower()}"] = df_regime_csv["markets_reporting_by_district"].apply(lambda d: d.get(dist, 0))
df_regime_csv.drop(columns=["markets_reporting_by_district"], inplace=True)
df_regime_csv.to_csv("regime_diagnostics_monthly.csv", index=False)
print(f"Exported regime_diagnostics_monthly.csv ({len(df_regime_csv)} months)")

regime_summary = {
    "total_months_analyzed": len(df_regime),
    "historical_regime_pre_2025_11": {
        "mean_monthly_rows": round(float(df_regime[df_regime["year_month"] < "2025-11"]["total_rows"].mean()), 1),
        "apmc_suffix_share": round(float(df_regime[df_regime["year_month"] < "2025-11"]["suffix_apmc_share"].mean()), 4),
        "fractional_price_share": round(float(df_regime[df_regime["year_month"] < "2025-11"]["fractional_price_share"].mean()), 4)
    },
    "transition_regime_2025_11_to_2026_02": {
        "months": ["2025-11", "2025-12", "2026-01", "2026-02"],
        "mean_monthly_rows": round(float(df_regime[(df_regime["year_month"] >= "2025-11") & (df_regime["year_month"] <= "2026-02")]["total_rows"].mean()), 1),
        "apmc_suffix_share": round(float(df_regime[(df_regime["year_month"] >= "2025-11") & (df_regime["year_month"] <= "2026-02")]["suffix_apmc_share"].mean()), 4),
        "fractional_price_share": round(float(df_regime[(df_regime["year_month"] >= "2025-11") & (df_regime["year_month"] <= "2026-02")]["fractional_price_share"].mean()), 4)
    },
    "post_migration_fractional_regime_2026_03_to_2026_09": {
        "mean_monthly_rows": round(float(df_regime[df_regime["year_month"] >= "2026-03"]["total_rows"].mean()), 1),
        "apmc_suffix_share": round(float(df_regime[df_regime["year_month"] >= "2026-03"]["suffix_apmc_share"].mean()), 4),
        "fractional_price_share": round(float(df_regime[df_regime["year_month"] >= "2026-03"]["fractional_price_share"].mean()), 4)
    }
}
with open("regime_diagnostics_summary.json", "w") as f:
    json.dump(regime_summary, f, indent=2)
print("Saved regime_diagnostics_summary.json")

print("\n=== REGIME DIAGNOSTICS SUMMARY ===")
print("Pre-Nov 2025:", regime_summary["historical_regime_pre_2025_11"])
print("Nov 2025 - Feb 2026 Transition:", regime_summary["transition_regime_2025_11_to_2026_02"])
print("Post-March 2026 Fractional Regime:", regime_summary["post_migration_fractional_regime_2026_03_to_2026_09"])


In [ ]:
# Cell 6: Part 5 - Date-Alignment Test for the 77 Differences across [d-3..d+3]
diff_77_path = "experiments/outputs/005/diff_77_deep_analysis.csv"
if not os.path.exists(diff_77_path):
    cands_77 = glob.glob("**/diff_77_deep_analysis.csv", recursive=True)
    if cands_77:
        diff_77_path = cands_77[0]

if os.path.exists(diff_77_path):
    df_diff77 = pd.read_csv(diff_77_path)
else:
    df_diff77 = merged_excel_v2[merged_excel_v2["price_diff"] != 0].copy()

print(f"Loaded {len(df_diff77)} discrepant rows for date-alignment evaluation.")

# Build dictionary for fast series-date lookup on v2 panel
v2_lookup = {}
for _, row in df_v2.iterrows():
    key = (row["District"], row["Market"], row["Commodity"], row["Variety"], row["Grade"])
    dt_str = pd.to_datetime(row["Arrival_Date"]).strftime("%Y-%m-%d")
    if key not in v2_lookup:
        v2_lookup[key] = {}
    v2_lookup[key][dt_str] = float(row["Modal_Price"])

date_alignment_results = []
shifts_tested = [-3, -2, -1, 0, 1, 2, 3]

for _, row in df_diff77.iterrows():
    raw_m = row.get("Market", row.get("market_raw", ""))
    canon_m = alias_map.get(raw_m, raw_m)
            
    dist = row.get("District_Std", row.get("District", ""))
    comm = row.get("Commodity_Std", row.get("Commodity", ""))
    var = row["Variety"]
    grd = row["Grade"]
    d_center = pd.to_datetime(row["Arrival_Date"])
    excel_price = float(row.get("Modal_Price_Clean_excel", row.get("Modal_Price", 0)))
    
    key = (dist, canon_m, comm, var, grd)
    series_dates = v2_lookup.get(key, {})
    
    matches_found = {}
    for k in shifts_tested:
        target_d = (d_center + pd.Timedelta(days=k)).strftime("%Y-%m-%d")
        if target_d in series_dates:
            panel_p = series_dates[target_d]
            matches_found[f"match_d{k:+d}"] = bool(abs(panel_p - excel_price) < 0.01)
        else:
            matches_found[f"match_d{k:+d}"] = False
            
    any_match_window = any(matches_found.values())
    exact_shift = None
    for k in shifts_tested:
        if matches_found[f"match_d{k:+d}"]:
            exact_shift = k
            break
            
    date_alignment_results.append({
        "district": dist,
        "market": canon_m,
        "commodity": comm,
        "variety": var,
        "grade": grd,
        "arrival_date": d_center.strftime("%Y-%m-%d"),
        "excel_price": excel_price,
        "panel_price_d0": series_dates.get(d_center.strftime("%Y-%m-%d")),
        "any_match_in_window_d3": any_match_window,
        "matched_shift_day": exact_shift,
        **matches_found
    })

df_date_alignment = pd.DataFrame(date_alignment_results)
df_date_alignment.to_csv("diff_77_date_alignment.csv", index=False)
print(f"Exported diff_77_date_alignment.csv ({len(df_date_alignment)} rows)")

shift_counts = {f"shift_{k:+d}": int(df_date_alignment[f"match_d{k:+d}"].sum()) for k in shifts_tested}
any_match_count = int(df_date_alignment["any_match_in_window_d3"].sum())

alignment_summary = {
    "total_discrepancies_tested": len(df_date_alignment),
    "matched_exact_on_d0": shift_counts["shift_+0"],
    "any_match_within_pm_3_days": any_match_count,
    "any_match_rate_pct": round(any_match_count / len(df_date_alignment) * 100, 2),
    "shift_breakdown": shift_counts
}
with open("diff_77_date_alignment_summary.json", "w") as f:
    json.dump(alignment_summary, f, indent=2)
print("Saved diff_77_date_alignment_summary.json")

print("\n=== DATE-ALIGNMENT TEST RESULTS (d-3..d+3) ===")
print("Alignment Summary:", alignment_summary)


In [ ]:
# Cell 7: Part 6 - Train-Only Eligibility & Rolling-Origin Feasibility on v2
# 1. Standard Training Window (<= 2025-12-18)
train_cutoff = "2025-12-18"
df_v2["Arrival_Date_dt"] = pd.to_datetime(df_v2["Arrival_Date"])
df_v2_train = df_v2[df_v2["Arrival_Date"] <= train_cutoff].copy()

train_metrics_v2 = []
for sid, grp in df_v2_train.groupby("series_id"):
    dts = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n_obs = len(grp)
    first_dt = dts[0]
    last_dt = dts[-1]
    span = (last_dt - first_dt).days + 1
    cov = n_obs / span if span > 0 else 0.0
    
    train_metrics_v2.append({
        "series_id": sid,
        "district": grp["District"].iloc[0],
        "market": grp["Market"].iloc[0],
        "commodity": grp["Commodity"].iloc[0],
        "variety": grp["Variety"].iloc[0],
        "grade": grp["Grade"].iloc[0],
        "n_obs_train": n_obs,
        "span_train": span,
        "cov_train": round(cov, 4),
        "last_date_train": last_dt.strftime("%Y-%m-%d"),
        "recency_days_at_cutoff": (pd.Timestamp(train_cutoff) - last_dt).days
    })

df_train_tiers_v2 = pd.DataFrame(train_metrics_v2)
df_train_tiers_v2["is_tier_a"] = (df_train_tiers_v2["n_obs_train"] >= 60) & (df_train_tiers_v2["cov_train"] >= 0.3)
df_train_tiers_v2["is_tier_b"] = (df_train_tiers_v2["n_obs_train"] >= 120) & (df_train_tiers_v2["cov_train"] >= 0.3)

tier_a_v2 = df_train_tiers_v2[df_train_tiers_v2["is_tier_a"]]
tier_b_v2 = df_train_tiers_v2[df_train_tiers_v2["is_tier_b"]]

print("=== TRAIN-ONLY ELIGIBILITY (v2 Panel, Cutoff <= 2025-12-18) ===")
print(f"Tier A Series Count (obs>=60, cov>=0.3): {len(tier_a_v2):,} across {tier_a_v2['market'].nunique()} markets (Total Obs: {tier_a_v2['n_obs_train'].sum():,})")
print(f"Tier B Series Count (obs>=120, cov>=0.3): {len(tier_b_v2):,} across {tier_b_v2['market'].nunique()} markets (Total Obs: {tier_b_v2['n_obs_train'].sum():,})")

# Focus crops breakdown
df_train_tiers_v2["is_focus"] = df_train_tiers_v2["commodity"].str.contains("Wheat|Potato|Onion|Tomato|Paddy|Rice", case=False, na=False)
focus_train_breakout = []
for (dist, mkt, comm), grp in df_train_tiers_v2[df_train_tiers_v2["is_focus"]].groupby(["district", "market", "commodity"]):
    focus_train_breakout.append({
        "district": dist,
        "market": mkt,
        "commodity": comm,
        "total_series": len(grp),
        "tier_a_series": int(grp["is_tier_a"].sum()),
        "tier_b_series": int(grp["is_tier_b"].sum()),
        "tier_a_obs": int(grp[grp["is_tier_a"]]["n_obs_train"].sum()),
        "tier_b_obs": int(grp[grp["is_tier_b"]]["n_obs_train"].sum())
    })
df_focus_v2 = pd.DataFrame(focus_train_breakout)
df_focus_v2.to_csv("focus_crops_train_eligibility_v2.csv", index=False)
df_train_tiers_v2.to_csv("train_eligibility_tiers_v2.csv", index=False)
print("Saved train_eligibility_tiers_v2.csv and focus_crops_train_eligibility_v2.csv")

# 2. Post-Migration Only Training Window (>= 2025-11-07)
post_mig_start = "2025-11-07"
df_v2_post_mig = df_v2[df_v2["Arrival_Date"] >= post_mig_start].copy()
post_mig_series = []
for sid, grp in df_v2_post_mig.groupby("series_id"):
    dts = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n_obs = len(grp)
    first_dt = dts[0]
    last_dt = dts[-1]
    span = (last_dt - first_dt).days + 1
    cov = n_obs / span if span > 0 else 0.0
    post_mig_series.append({
        "series_id": sid,
        "n_obs": n_obs,
        "span": span,
        "cov": round(cov, 4),
        "is_tier_a": (n_obs >= 60) and (cov >= 0.3),
        "is_tier_b": (n_obs >= 120) and (cov >= 0.3)
    })
df_post_mig = pd.DataFrame(post_mig_series)
print(f"\n=== POST-MIGRATION ONLY WINDOW (>= {post_mig_start}) ===")
print(f"Total Observations: {len(df_v2_post_mig):,}")
print(f"Total Active Series: {len(df_post_mig):,}")
print(f"Series meeting Tier A (>=60 obs, cov>=0.3): {df_post_mig['is_tier_a'].sum():,}")
print(f"Series meeting Tier B (>=120 obs, cov>=0.3): {df_post_mig['is_tier_b'].sum():,}")

# 3. Rolling-Origin Cross-Validation Feasibility on v2 (4 Expanding Folds, 7-date embargo)
end_date = pd.Timestamp("2026-09-30")
cur_end = end_date
folds_def = []

for fold_num in [4, 3, 2, 1]:
    t_end = cur_end
    t_start = t_end - pd.Timedelta(days=59) # 60 days
    emb_end = t_start - pd.Timedelta(days=1)
    emb_start = t_start - pd.Timedelta(days=7) # 7 dates
    tr_end = emb_start - pd.Timedelta(days=1)
    
    disc_start = pd.Timestamp("2026-02-22")
    disc_end = pd.Timestamp("2026-03-31")
    overlaps_test = bool((t_start <= disc_end) and (t_end >= disc_start))
    overlaps_train = bool((tr_end >= disc_start))
    
    folds_def.append({
        "fold": fold_num,
        "train_start": "2021-09-01",
        "train_end": tr_end.strftime("%Y-%m-%d"),
        "embargo_start": emb_start.strftime("%Y-%m-%d"),
        "embargo_end": emb_end.strftime("%Y-%m-%d"),
        "embargo_dates_count": 7,
        "test_start": t_start.strftime("%Y-%m-%d"),
        "test_end": t_end.strftime("%Y-%m-%d"),
        "test_days_count": 60,
        "test_overlaps_feb_mar_discrepancy": overlaps_test,
        "train_overlaps_feb_mar_discrepancy": overlaps_train
    })
    cur_end = t_start - pd.Timedelta(days=1)

folds_def = sorted(folds_def, key=lambda x: x["fold"])
tier_a_ids_v2 = set(tier_a_v2["series_id"])
tier_b_ids_v2 = set(tier_b_v2["series_id"])

rolling_results_v2 = []
for f in folds_def:
    t_s = f["test_start"]
    t_e = f["test_end"]
    
    fold_test_df = df_v2[(df_v2["Arrival_Date"] >= t_s) & (df_v2["Arrival_Date"] <= t_e)]
    series_origins = fold_test_df.groupby("series_id").size().to_dict()
    
    tier_a_counts = [series_origins.get(sid, 0) for sid in tier_a_ids_v2]
    tier_b_counts = [series_origins.get(sid, 0) for sid in tier_b_ids_v2]
    
    tier_a_ge_20 = sum(1 for c in tier_a_counts if c >= 20)
    tier_b_ge_20 = sum(1 for c in tier_b_counts if c >= 20)
    
    rolling_results_v2.append({
        "fold": f["fold"],
        "train_window": f"{f['train_start']} to {f['train_end']}",
        "embargo_window": f"{f['embargo_start']} to {f['embargo_end']}",
        "test_window": f"{f['test_start']} to {f['test_end']}",
        "test_overlaps_feb_mar_discrepancy": f["test_overlaps_feb_mar_discrepancy"],
        "train_overlaps_feb_mar_discrepancy": f["train_overlaps_feb_mar_discrepancy"],
        "tier_a_total_series": len(tier_a_ids_v2),
        "tier_a_series_ge_20_origins": tier_a_ge_20,
        "tier_a_ge_20_pct": round(tier_a_ge_20 / len(tier_a_ids_v2) * 100, 2),
        "tier_b_total_series": len(tier_b_ids_v2),
        "tier_b_series_ge_20_origins": tier_b_ge_20,
        "tier_b_ge_20_pct": round(tier_b_ge_20 / len(tier_b_ids_v2) * 100, 2)
    })

print("\n=== ROLLING-ORIGIN FEASIBILITY SUMMARY (v2 Panel, 4 Folds) ===")
df_rolling_v2 = pd.DataFrame(rolling_results_v2)
print(df_rolling_v2.to_string(index=False))

with open("rolling_origin_feasibility_v2.json", "w") as f:
    json.dump({
        "folds": rolling_results_v2,
        "post_migration_training_feasibility": {
            "window": f"{post_mig_start} to 2026-09-30",
            "total_observations": len(df_v2_post_mig),
            "tier_a_series": int(df_post_mig['is_tier_a'].sum()),
            "tier_b_series": int(df_post_mig['is_tier_b'].sum())
        }
    }, f, indent=2)
print("Saved rolling_origin_feasibility_v2.json")


In [ ]:
# Cell 8: Part 7 - Final Output Artifact Verification & Hashes
print("=== GENERATED ARTIFACTS VERIFICATION ===")
expected_artifacts = [
    "market_inventory_raw.csv",
    "market_alias_candidates.csv",
    "market_aliases_accepted.csv",
    "market_aliases_review.csv",
    "pipeline_disposition_summary.json",
    "panel_v1_vs_v2_comparison.csv",
    "corroboration_v2_summary.json",
    "regime_diagnostics_monthly.csv",
    "regime_diagnostics_summary.json",
    "diff_77_date_alignment.csv",
    "diff_77_date_alignment_summary.json",
    "train_eligibility_tiers_v2.csv",
    "focus_crops_train_eligibility_v2.csv",
    "rolling_origin_feasibility_v2.json",
    "processing_manifest_v2.json"
]

manifest_outputs = []
for fname in expected_artifacts:
    if os.path.exists(fname):
        sz = os.path.getsize(fname)
        sha = compute_sha256(fname)
        line_cnt = None
        if fname.endswith(".csv"):
            with open(fname) as f:
                line_cnt = sum(1 for _ in f)
        manifest_outputs.append({
            "artifact": fname,
            "size_bytes": sz,
            "sha256": sha,
            "line_count": line_cnt
        })
        print(f"FILE: {fname:<36} | {sz:>10,} bytes | lines: {str(line_cnt):<8} | sha256: {sha}")
    else:
        print(f"MISSING: {fname}")

# Also record v2 panel checksum if present
if os.path.exists("mandi_5year_validated_v2.csv"):
    v2_sz = os.path.getsize("mandi_5year_validated_v2.csv")
    v2_sha = compute_sha256("mandi_5year_validated_v2.csv")
    print(f"\nPANEL V2: mandi_5year_validated_v2.csv | {v2_sz:,} bytes | sha256: {v2_sha}")
    manifest_outputs.append({
        "artifact": "mandi_5year_validated_v2.csv",
        "size_bytes": v2_sz,
        "sha256": v2_sha,
        "line_count": None
    })

with open("kernel_execution_manifest.json", "w") as f:
    json.dump({"artifacts": manifest_outputs, "timestamp_utc": time.time()}, f, indent=2)
print("\nKernel Execution Completed Successfully!")
